<a href="https://colab.research.google.com/github/leticiasdrummond/Modelos-Base/blob/main/battery_scheduling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Relatório Técnico: Otimização de Despacho de Baterias (BESS)

Este documento integra a fundamentação teórica de Sistemas de Gestão de Energia (EMS) com a implementação numérica utilizando o solver Gurobi.

## 1. Mapeamento de Referencial Teórico e Metodologia

| Conceito Principal | Base Literária | Implementação Numérica |
| :--- | :--- | :--- |
| **Gestão de Energia (EMS)** | Bertsimas et al. (Otimização Robusta) | Funções `build_lgs` e `build_s` |
| **Balanço de Energia** | Lei de Kirchhoff | Restrição `meter_balance` |
| **Degradação de Lítio** | Han et al. (2014) / Modelo Power-law | Função objetivo com penalidade $\beta$ |

---

## 2. Parâmetros do Modelo

Definimos parâmetros realistas para um ciclo de 24 horas.

**Referencial:** Parâmetros de eficiência baseados em sistemas comerciais de íon-lítio ($̗ ≈ 96\%$).

In [1]:
%pip install gurobipy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.9/14.9 MB 45.0 MB/s eta 0:00:00


In [22]:
# Importar pacotes necessários
import math
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from gurobipy import GRB

# === CONFIGURAÇÃO DA INSTÂNCIA ===

# Horizonte de tempo
T = 24
dt_h = 1.0  # horas por passo

# Parâmetros de tarifa e custo
beta = 1.5                # expoente da lei de potência para degradação não-linear
kappa_cyc = 1              # €/EFC (custo por ciclo completo equivalente)

# Parâmetros da Bateria
E_max_kwh = 13.5
P_ch_max_kw = 5.0
P_dis_max_kw = 5.0
eta_ch = 0.96
eta_dis = 0.96
SoC0_kwh = 6.0
SoC_terminal = 6.0

### Seção 1: Geração de Perfis e Dados de Entrada
**Conceito:** Modelagem da Demanda Líquida ($NetLoad$).
**Fórmula:** $NetLoad_t = L_t - G_t$

Aqui definimos os vetores de carga, geração PV e os sinais de preço da rede para o horizonte de 24h.

In [23]:
# === GERAÇÃO DA INSTÂNCIA ===
def default_24h_profiles():
    # Perfis de carga (load), solar (pv), preço de compra (imp) e venda (exp)
    load_24 = [1.4,1.4,1.3,1.2,1.1,1.0, 0.6,0.6,0.7,0.8,1.0,1.2,
               1.3,1.4,1.5,1.4,1.2,1.0, 1.1,1.3,1.6,1.8,1.8,1.6]
    pv_24   = [0.0,0.0,0.0,1.0,2.0,3.0, 0.2,0.5,1.0,2.0,3.2,3.8,
               4.0,4.2,3.6,2.8,1.8,0.8, 2.2,1.1,0.6,0.0,0.0,0.0]
    imp_24  = [0.16,0.15,0.14,0.14,0.15,0.17, 0.18,0.17,0.16,0.16,0.15,0.15,
               0.16,0.18,0.20,0.22,0.24,0.26, 0.30,0.34,0.38,0.40,0.38,0.34]
    exp_24  = [0.1,0.1,0.1,0.1,0.1,0.1, 0.1,0.1,0.1,0.1,0.1,0.1,
               0.14,0.16,0.18,0.20,0.22,0.24, 0.29,0.3,0.3,0.4,0.3,0.3]
    return load_24, pv_24, imp_24, exp_24

load, pv, imp, exp = default_24h_profiles()

# Pré-visualização dos dados do primeiro dia
df_preview = pd.DataFrame({
    "t": list(range(24)),
    "Carga_kW": load[:24],
    "PV_kW": pv[:24],
    "Preço_Compra_€/kWh": imp[:24],
    "Preço_Venda_€/kWh": exp[:24]
})
display(df_preview.head(24))

,t,Carga_kW,PV_kW,Preço_Compra_€/kWh,Preço_Venda_€/kWh
0,0,1.4,0.0,0.16,0.10
1,1,1.4,0.0,0.15,0.10
2,2,1.3,0.0,0.14,0.10
3,3,1.2,1.0,0.14,0.10
4,4,1.1,2.0,0.15,0.10
5,5,1.0,3.0,0.17,0.10
6,6,0.6,0.2,0.18,0.10
7,7,0.6,0.5,0.17,0.10
8,8,0.7,1.0,0.16,0.10
9,9,0.8,2.0,0.16,0.10


## 3. Implementação dos Modelos Numéricos

### A. Modelo Carga & Geração (LG)
**Conceito:** Baseline sem armazenamento. A rede absorve todo o excedente ou supre o déficit.

### B. Modelo de Armazenamento (S)
**Conceito:** Arbitragem de preço pura. Compra na tarifa baixa, vende na tarifa alta.

### C. Modelo Carga, Geração & Armazenamento (LGS)
**Conceito:** Integração completa Behind-the-Meter (BTM) com penalidade por degradação.

### Seção 2: Modelo de Referência (Baseline LG)
**Conceito:** Operação passiva sem armazenamento.
**Teoria:** Toda a demanda líquida é suprida ou absorvida pela rede elétrica (Grid).

### Cálculo de Lucro e Balanço Financeiro (LG)

**Conceito Principal:** Fluxo de Caixa Operacional Líquido ($Profit$).  
**Referencial:** Princípios de Contabilidade de Energia (*Energy Accounting*).

**Formulação Matemática:**
$$\text{Profit} = \sum_{t=1}^{T} \left[ (\text{Preço Export}_{t} \cdot P_{exp,t} \cdot \Delta t) - (\text{Preço Import}_{t} \cdot P_{imp,t} \cdot \Delta t) \right]$$

**Parâmetros e Unidades de Medida:**

1.  **Grandezas Físicas e Temporais:**
    *   **$T$ (24):** Horizonte de simulação representando um ciclo diário completo.
    *   **$\Delta t$ ou `dt_h` (1.0 h):** Passo de tempo (*Time Step*). Define que os cálculos de energia ocorrem a cada 1 hora.
    *   **Potência ($P$):** Medida em Quilowatts (**kW**). Representa a taxa instantânea de fluxo de energia.
    *   **Energia ($P \cdot \Delta t$):** Medida em Quilowatt-hora (**kWh**). É a integração da potência no tempo.

2.  **Especificações Técnicas (Hardware):**
    *   **$E_{max}$ (kWh):** Capacidade nominal total de armazenamento da bateria.
    *   **$P_{max}$ (kW):** Limite físico de carga e descarga do inversor.
    *   **$\eta$ (adimensional):** Eficiência (ex: 0.96 para 96%). Representa perdas por conversão térmica.

3.  **Grandezas Econômicas:**
    *   **Preços:** Medidos em Euros por kWh (**€/kWh**).
    *   **Lucro:** Medido em Euros (**€**).
    *   **$\kappa_{cyc}$ (€/EFC):** Custo de degradação por ciclo completo equivalente.
    *   **$\beta$ (adimensional):** Coeficiente de severidade para modelagem de custo não-linear.

In [24]:
# Para o modelo de Carga e Geração (LG), o lucro é calculado simplesmente pela
# demanda/excedente líquido de energia multiplicado pelo preço de importação/exportação.

def build_lg(load_kw, pv_kw, import_price, export_price, dt_h):
    # Configuração do tempo e perfis
    T = len(load_kw)
    P_imp0, P_exp0 = [], []

    # Calcular custos de importação/exportação para cada passo de tempo
    for t in range(T):
        net = load_kw[t] - pv_kw[t]  # calcular déficit/excedente de potência
        P_imp0.append(max(0.0, net))
        P_exp0.append(max(0.0, -net))

    # Calcular lucro acumulando custos ao longo do horizonte
    profit = 0.0
    for t in range(T):
        imp_cost = import_price[t] * P_imp0[t] * dt_h
        exp_rev  = export_price[t] * P_exp0[t] * dt_h
        profit += (exp_rev - imp_cost)

    return profit, P_imp0, P_exp0

### Seção 3: Modelo de Arbitragem Pura (S)
**Conceito:** Energy Arbitrage (Arbitragem de Preço).




**Dinâmica do Estado de Carga (SoC):**
$E_t = E_{t-1} + (\eta_{ch} \cdot x_{ch} \cdot \Delta t) - (\frac{x_{dis}}{\eta_{dis}} \cdot \Delta t)$

### Parâmetros e Unidades de Medida

#### Detalhamento Técnico das Grandezas:

Abaixo, apresento o mapeamento detalhado dos parâmetros configurados no código e suas respectivas unidades de medida, fundamentais para a precisão do modelo físico e econômico:

**1. Parâmetros Temporais:**
*   $T$ (24): Horizonte de tempo da simulação, representando um ciclo diário completo.
*   $\Delta t$ ou `dt_h` (1.0 h): Passo de tempo (*Time Step*). Define o intervalo de integração para os cálculos de energia.

**2. Especificações da Bateria (BESS):**
*   $E_{max}$ (13.5 kWh): Capacidade nominal de energia. É o limite máximo de armazenamento do sistema.
*   $P_{ch,max} / P_{dis,max}$ (5.0 kW): Taxas de potência máxima para carga e descarga, limitando o fluxo instantâneo.
*   $\eta_{ch} / \eta_{dis}$ (0.96 ou 96%): Eficiência de ida e volta (*Round-trip efficiency*), contabilizando perdas térmicas.
*   $SoC_{0} / SoC_{terminal}$ (6.0 kWh): Estado de Carga inicial e final para garantir a ciclicidade do modelo.

**3. Parâmetros Econômicos e de Degradação:**
*   $\kappa_{cyc}$ (€/EFC): Custo por Ciclo Completo Equivalente. Mapeia o desgaste físico em valor monetário.
*   $\beta$ (1.5): Coeficiente de severidade. Expoente adimensional que modela a natureza não-linear da degradação.
*   **Preços** ($imp / exp$): Tarifas de mercado em **€/kWh** para compra e venda de energia da rede elétrica.

In [25]:
import gurobipy as gp
from gurobipy import GRB

def build_s(T, dt_h, import_price, export_price,
                        E_max_kwh, Pmax_kw, eta_ch, eta_dis, SoC0_kwh,
                        SoC_terminal=None):
    # 1. Criar o modelo de otimização
    m = gp.Model("s_model")

    # 2. Definir parâmetros básicos do solver
    m.Params.OutputFlag = 1 # 1 = imprime log; 0 = silencioso
    m.Params.TimeLimit = 10 # Limite de tempo em segundos

    # 3. Criar variáveis de decisão
    rng = range(T)

    # Potência de carga (kW), limitada entre 0 e Pmax_kw
    x_ch = m.addVars(rng, lb=0.0, ub=Pmax_kw, name="x_ch")
    # Potência de descarga (kW)
    x_dis = m.addVars(rng, lb=0.0, ub=Pmax_kw, name="x_dis")
    # Estado de Carga (SoC) em kWh, limitado pela capacidade máxima
    SoC = m.addVars(rng, lb=0.0, ub=E_max_kwh, name="SoC")

    # 4. Criar restrições: Balanço de SoC da bateria
    for t in rng:
        inflow  = eta_ch * x_ch[t] * dt_h          # energia entrando
        outflow = (1.0 / eta_dis) * x_dis[t] * dt_h  # energia saindo

        if t == 0:
            m.addConstr(SoC[t] == SoC0_kwh + inflow - outflow, name=f"balanco_soc_{t}")
        else:
            m.addConstr(SoC[t] == SoC[t-1] + inflow - outflow, name=f"balanco_soc_{t}")

    # Restrição de SoC terminal opcional
    if SoC_terminal is not None:
        m.addConstr(SoC[T - 1] >= SoC_terminal, name="soc_terminal")

    # 5. Função Objetivo: Maximização do Lucro (Arbitragem)
    profit = gp.quicksum(
        (export_price[t] * x_dis[t] - import_price[t] * x_ch[t]) * dt_h
        for t in rng
    )
    m.setObjective(profit, GRB.MAXIMIZE)

    return m, {"x_ch": x_ch, "x_dis": x_dis, "SoC": SoC}

### Seção 4: Integração Behind-the-Meter e Degradação (LGS)
**Conceito:** Otimização Multi-objetivo BTM.
**Modelo de Degradação:**
$Custo = \kappa \cdot (\frac{\sum Fluxo}{E_{max}})^{\beta}$

Neste modelo, o Gurobi busca o equilíbrio entre o lucro comercial e o desgaste físico da bateria.

In [26]:
import gurobipy as gp
from gurobipy import GRB

def build_lgs(T, dt_h, load_kw, pv_kw, import_price, export_price,
                    E_max_kwh, P_ch_max_kw, P_dis_max_kw, eta_ch, eta_dis, SoC0_kwh,
                    kappa_cyc, beta, SoC_terminal=None):
    # 1. Inicializar o modelo
    m = gp.Model("lgs_model")
    m.Params.OutputFlag = 1
    m.Params.TimeLimit = 10

    # 2. Variáveis de decisão
    rng = range(T)
    x_ch = m.addVars(rng, lb=0.0, ub=P_ch_max_kw, name="x_ch")
    x_dis = m.addVars(rng, lb=0.0, ub=P_dis_max_kw, name="x_dis")
    SoC = m.addVars(rng, lb=0.0, ub=E_max_kwh, name="SoC")
    P_imp = m.addVars(rng, lb=0.0, name="P_imp") # Importação da rede
    P_exp = m.addVars(rng, lb=0.0, name="P_exp") # Exportação para rede

    # 3. Restrições de Balanço da Bateria
    for t in rng:
        inflow  = eta_ch * x_ch[t] * dt_h
        outflow = (1.0 / eta_dis) * x_dis[t] * dt_h
        if t == 0:
            m.addConstr(SoC[t] == SoC0_kwh + inflow - outflow, name=f"soc_bal_{t}")
        else:
            m.addConstr(SoC[t] == SoC[t-1] + inflow - outflow, name=f"soc_bal_{t}")

    # 4. Restrição de Balanço Energético (Ponto de Conexão)
    # P_imp - P_exp + x_dis - x_ch = Carga - PV
    for t in rng:
        rhs = load_kw[t] - pv_kw[t]
        m.addConstr(P_imp[t] - P_exp[t] + x_dis[t] - x_ch[t] == rhs, name=f"medidor_{t}")

    if SoC_terminal is not None:
        m.addConstr(SoC[T - 1] >= SoC_terminal, name="soc_final")

    # 5. Função Objetivo Parte 1: Lucro Comercial
    comercial_profit = gp.quicksum((export_price[t] * P_exp[t] - import_price[t] * P_imp[t]) * dt_h for t in rng)

    # 6. Função Objetivo Parte 2: Custo de Degradação Não-Linear
    c_flow_pow = m.addVar(lb=0.0, name="fluxo_degr_pow")
    m.addConstr(c_flow_pow == gp.quicksum((dt_h * (x_ch[t] + x_dis[t]) / E_max_kwh) ** beta for t in rng))
    deg_cost = kappa_cyc * c_flow_pow

    m.setObjective(comercial_profit - deg_cost, GRB.MAXIMIZE)

    return m, {"x_ch": x_ch, "x_dis": x_dis, "SoC": SoC, "P_imp": P_imp, "P_exp": P_exp}

## 4. Execução da Otimização

Nesta etapa, o Gurobi processa as restrições lineares e a função objetivo não-linear (degradação) para encontrar o despacho ótimo.

In [27]:
# CONFIGURAÇÃO E EXECUÇÃO DO MODELO

# Escolha a variante do modelo:
# LG = Apenas Carga e Geração | S = Apenas Armazenamento | LGS = Completo
modelVariant = "S"

try:
    if modelVariant == "LG":
        base_profit, P_imp0, P_exp0 = build_lg(load, pv, imp, exp, dt_h)
        print(f"Lucro Base (sem bateria) = {base_profit:.2f} €")

    elif modelVariant == "S":
        m, v = build_s(T, dt_h, imp, exp, E_max_kwh, P_ch_max_kw, eta_ch, eta_dis, SoC0_kwh, SoC_terminal)
        m.optimize()

    elif modelVariant == "LGS":
        m, v = build_lgs(T, dt_h, load, pv, imp, exp, E_max_kwh, P_ch_max_kw, P_dis_max_kw, eta_ch, eta_dis, SoC0_kwh, kappa_cyc, beta, SoC_terminal)
        m.optimize()

    # Verificação de Status do Gurobi
    if modelVariant in ("S", "LGS"):
        if m.Status == GRB.OPTIMAL:
            print(f"Status: Otimização concluída. Lucro total: {m.ObjVal:.2f} €")
        else:
            print(f"O solver parou com status: {m.Status}")

except gp.GurobiError as e:
    print(f"Erro no Gurobi: {e.message}")

Set parameter OutputFlag to value 1
Set parameter TimeLimit to value 10
Gurobi Optimizer version 13.0.2 build v13.0.2rc1 (linux64 - "Ubuntu 22.04.5 LTS")

CPU model: AMD EPYC 7B12, instruction set [SSE2|AVX|AVX2]
Thread count: 1 physical cores, 2 logical processors, using up to 2 threads

Non-default parameters:
TimeLimit  10

Optimize a model with 25 rows, 72 columns and 96 nonzeros (Max)
Model fingerprint: 0xdab46bf5
Model has 48 linear objective coefficients
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  Objective range  [1e-01, 4e-01]
  Bounds range     [5e+00, 1e+01]
  RHS range        [6e+00, 6e+00]

Presolve removed 2 rows and 2 columns
Presolve time: 0.01s
Presolved: 23 rows, 70 columns, 92 nonzeros

Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    2.1150000e+01   1.232628e+02   0.000000e+00      0s
       6    1.5662500e+00   0.000000e+00   0.000000e+00      0s

Solved in 6 iterations and 0.01 seconds (0.00 work units)
Optimal objective 

## 5. Resultados e Visualizações

Extraímos as variáveis de decisão para analisar o comportamento do Estado de Carga (SoC) em relação aos sinais de preço da rede.

# Relatório de Revisão de Literatura e Implementação Numérica

Este documento correlaciona os conceitos teóricos de gestão de energia com a implementação prática no código.

## 1. Mapeamento Conceitual

| Conceito Principal | Referencial Teórico | Implementação no Código |
| :--- | :--- | :--- |
| **Despacho de Bateria** | Programação Linear (EMS) | Função `build_lgs` e `build_s` |
| **Balanço de Energia** | Lei de Kirchhoff | Restrição `meter_balance` |
| **Degradação** | Modelagem Não-Linear | Função objetivo com `beta` |

## 2. Formulações Matemáticas e Código

### A. Balanço de Energia (Behind-the-Meter)
**Teoria:** A soma das potências de entrada e saída em um nó deve ser zero.
**Fórmula:** $P_{imp} - P_{exp} + x_{dis} - x_{ch} = L_t - G_t$

```python
# Implementação da Restrição de Balanço
for t in rng:
    rhs = load_kw[t] - pv_kw[t]
    m.addConstr(
        P_imp[t] - P_exp[t] + x_dis[t] - x_ch[t] == rhs,
        name=f"meter_balance_{t}"
    )
```

### B. Dinâmica do Estado de Carga (SoC)
**Teoria:** O SoC atual é o estado anterior mais a energia líquida (considerando eficiências).
**Fórmula:** $E_t = E_{t-1} + (\eta_{ch} \cdot x_{ch} \cdot \Delta t) - (\frac{x_{dis}}{\eta_{dis}} \cdot \Delta t)$

```python
# Implementação da Equação de Diferenças
inflow  = eta_ch * x_ch[t] * dt_h
outflow = (1.0 / eta_dis) * x_dis[t] * dt_h
m.addConstr(SoC[t] == SoC[t-1] + inflow - outflow)
```

### C. Modelo de Custo de Degradação
**Teoria:** Penalização baseada no fluxo de energia (C-flow) usando uma lei de potência.
**Fórmula:** $Custo = \kappa \cdot (\frac{\text{Fluxo}}{E_{max}})^{\beta}$

```python
# Implementação Não-Linear
m.addConstr(c_flow_pow == gp.quicksum((dt_h * (x_ch[t] + x_dis[t]) / E_max_kwh) ** beta for t in rng))
profit -= kappa_cyc * c_flow_pow
```

---
*Este mapa serve como base para a fundamentação teórica do projeto de otimização.*

In [21]:
### Seção 5: Conclusões e Análise de Resultados



SyntaxError: invalid syntax (985125821.py, line 3)

Os resultados demonstram que a inclusão de restrições físicas (eficiência) e econômicas (degradação) altera significativamente a estratégia de despacho em comparação com modelos puramente financeiros.

## 6. Conclusões e Insights

O modelo demonstra que a bateria atua como um 'buffer', suavizando a demanda líquida e permitindo a arbitragem financeira, resultando em um aumento do lucro operacional diário.